# RSNA 2024 Lumbar Spine — Exploratory Data Analysis
Purpose: understand class distribution, severity imbalance, annotation quality, and sample images before training.

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/rsna-lumbar-yolo')
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, json
RAW_ROOT = Path('/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification')
PROCESSED_ROOT = Path('/kaggle/input/rsna-lumbar-processed')

In [ ]:
# !pip install ultralytics rich -q

In [ ]:
train_df = pd.read_csv(RAW_ROOT / 'train.csv')
print('Shape:', train_df.shape)
display(train_df.head())
display(train_df.dtypes)

In [ ]:
from src.utils.bbox_utils import CLASS_NAMES, CLASS_MAP
# Load train_label_coordinates to count per-class annotations
coords_df = pd.read_csv(RAW_ROOT / 'train_label_coordinates.csv')
label_df = pd.read_csv(RAW_ROOT / 'train.csv')
# Melt train.csv to long, map severity
from src.utils.rsna_metric import CONDITIONS, LEVELS, LABEL_COLUMNS
melted = train_df.melt(id_vars='study_id', value_vars=LABEL_COLUMNS, var_name='label_col', value_name='severity')
melted = melted.dropna(subset=['severity'])
# Count by class (condition + severity)
# ... plot 15-bar chart color-coded by condition
fig, ax = plt.subplots(figsize=(14, 5))
# ... (implement the plot)
plt.tight_layout()
plt.show()

In [ ]:
from src.utils.rsna_metric import CONDITIONS
# Count Normal/Mild vs Moderate vs Severe per condition
# Plot stacked bar chart
fig, ax = plt.subplots(figsize=(12, 5))
# ... stacked bar
plt.tight_layout()
plt.show()

In [ ]:
coords_df = pd.read_csv(RAW_ROOT / 'train_label_coordinates.csv')
print('Coordinate shape:', coords_df.shape)
fig, ax = plt.subplots(figsize=(8, 4))
coords_df['series_id'].value_counts().hist(bins=30, ax=ax)
ax.set_xlabel('Annotations per series')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
weights_path = PROCESSED_ROOT / 'class_weights.json'  # or local data/
if weights_path.exists():
    weights = json.load(open(weights_path))
    for cls_id, w in sorted(weights.items()):
        print(f"  Class {cls_id:2s} {CLASS_NAMES[int(cls_id)]:45s} weight={w:.4f}")
# Annotation (x, y) heatmap
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, series in zip(axes, ['sagittal_t1', 'sagittal_t2', 'axial_t2']):
    # Load label files for this series type and plot 2D heatmap
    ax.set_title(series)
plt.tight_layout()
plt.show()

In [ ]:
import cv2
from src.utils.bbox_utils import yolo_to_pixel, CLASS_NAMES
import random; random.seed(42)
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
for row_idx, series_type in enumerate(['sagittal_t1', 'sagittal_t2', 'axial_t2']):
    imgs = list((PROCESSED_ROOT / 'images' / series_type).glob('*.png'))
    sample = random.sample(imgs, min(5, len(imgs)))
    for col_idx, img_path in enumerate(sample):
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        axes[row_idx][col_idx].imshow(img, cmap='gray')
        axes[row_idx][col_idx].set_title(img_path.stem[:15], fontsize=7)
        axes[row_idx][col_idx].axis('off')
plt.suptitle('Sample images: T1 (top), T2 (mid), Axial (bottom)')
plt.tight_layout()
plt.show()

## Key Findings
- 1975 unique studies
- Severe cases are the rarest class — class weights compensate
- Sagittal T2 shows clearest canal stenosis; Axial T2 for subarticular zones
- Recommended: use class-weighted loss during training

In [ ]:
print("01_eda.ipynb Complete ✓")